# OLCF Resource Discovery & Job Submission via the AmSC Python Client

This notebook demonstrates how to query OLCF resources in the **open** and **moderate** enclaves, check an S3M token, and submit and manage compute jobs on OLCF resources (Defiant, Wombat, Quokka, Odo, Frontier, etc.) using the AmSC Python Client's facility integration over the S3M **IRI API v2**.

**What you'll do:**

1. Create an AmSC client
2. Provide your S3M tokens, connect to an OLCF enclave, confirm the client uses API v2, and explore compute resources
3. Check who your token belongs to (`whoami`)
4. Check for active incidents
5. Optionally submit a "hello world" job after enabling the default-off gate
6. Optionally monitor the submitted job until completion

**Prerequisites:**

- `amsc-client` installed (`pip install -r requirements.txt`)
- A valid OLCF account and project allocation ([OLCF accounts and projects](https://docs.olcf.ornl.gov/accounts/index.html))
- An S3M token for each enclave you use, scoped to your project ([S3M token management](https://docs.olcf.ornl.gov/services_and_applications/s3m/overview.html#token-management))

## Authentication and validation status

- **Credential domain:** an OLCF **S3M token** (Project Access Token), one per enclave. It is separate from `AMSC_TOKEN`, which this notebook does not use.
- **Public endpoints:** facility info, resource listing and incidents need no token.
- **Token required:** checking your identity (`whoami`) and all job operations.
- **Supplying tokens:** Step 2 reads them from environment variables, from token files, or from a hidden prompt. Never paste a token into a notebook cell.
- **IRI v2:** `amsc-client 0.7.1` has no built-in OLCF configuration and generates IRI v1 routes. The repository helper `olcf_s3m` (`notebooks/olcf_s3m.py`) routes the client to S3M's v2 API for OLCF hosts only.
- **Validation status (2026-10-09):** live-validated by the tutorial maintainers on both enclaves for `whoami`, resource discovery, job listing, job launch and job status. Job update and cancel are not covered by this notebook.

**Not yet available in S3M:** Filesystem, Account (except `whoami`), Task, and Storage APIs.

In [ ]:
import os
import time
import amsc_client
print(amsc_client.version_info())
from amsc_client import Client, Resource, Job, ApiError

# OLCF S3M helpers (notebooks/olcf_s3m.py): routes amsc-client to the IRI v2 API
from olcf_s3m import (
    ENCLAVES, MissingTokenError, recent_requests, register_olcf,
    resolve_token, verify_api_version, whoami,
)

## Step 1: Create the AmSC Client

The client manages authentication and provides access to all AmSC services. For this tutorial, we only need facility access — no catalog auth is required.

In [ ]:
# Create a minimal client — we only need facility access, not catalog
client = Client(token="not-needed-for-facilities")
print("✅ Client created")

## Step 2: Connect to OLCF and Explore Resources

OLCF is not a built-in facility. `register_olcf()` registers an enclave with the client and points it at the S3M IRI v2 API:

| Enclave | Base URL | Resources |
|---------|----------|-----------|
| `open` | `https://amsc-open.s3m.olcf.ornl.gov` | Defiant, Wombat, Quokka, Odo |
| `moderate` | `https://amsc-moderate.s3m.olcf.ornl.gov` | Frontier |

> **Why a helper?** amsc-client currently generates IRI **v1** routes. S3M serves the IRI **v2** API, which also describes resource types differently. `olcf_s3m.register_olcf()` routes the client to v2 and translates those differences, only for OLCF hosts. The token is sent as `Authorization: Bearer <token>`, which S3M's IRI routes require.

### 2a. Provide your S3M tokens

Each enclave uses its own token. Set `TOKEN_SOURCE` in the next cell to one of:

| `TOKEN_SOURCE` | Where the token comes from |
|----------------|----------------------------|
| `"env"` | `OLCF_S3M_TOKEN_OPEN` / `OLCF_S3M_TOKEN_MODERATE` exported before starting Jupyter, or a `.env` file at the repository root |
| `"file"` | The files in `TOKEN_FILES`. Each holds either the bare token, or a line like `OLCF_S3M_TOKEN_OPEN=<token>` |
| `"prompt"` | A hidden box where you paste each token (press Enter to skip an enclave) |

Without a token you can still list resources and incidents.

> ⚠️ Don't paste a token directly into a code cell. Cell contents are saved in the `.ipynb` file and can end up in git. The `"prompt"` option keeps the token out of the notebook. Keep token files outside the repository, readable only by you (`chmod 600`).

In [ ]:
# ── Token and enclave settings ─────────────────────────────────────
TOKEN_SOURCE = "env"        # "env", "file", or "prompt"
TOKEN_FILES = {             # used when TOKEN_SOURCE = "file"
    "open": "~/.config/olcf/.env-open",
    "moderate": "~/.config/olcf/.env-moderate",
}
ACTIVE_ENCLAVE = os.environ.get("OLCF_ENCLAVE", "open")   # "open" or "moderate"
# ───────────────────────────────────────────────────────────────────

s3m_tokens = {}
for enclave in ENCLAVES:
    try:
        s3m_tokens[enclave] = resolve_token(
            enclave, TOKEN_SOURCE, token_file=TOKEN_FILES.get(enclave)
        )
    except MissingTokenError as exc:
        print(f"⚠️  {exc}")
        s3m_tokens[enclave] = None
    status = "loaded" if s3m_tokens[enclave] else "not set (public endpoints only)"
    print(f"{enclave:9s} token {status}")   # the token itself is never printed

### 2b. Register the enclaves

In [ ]:
olcf_facilities = {}
for enclave in ENCLAVES:
    # Public endpoints work without a token; a placeholder keeps the client usable.
    olcf_facilities[enclave] = register_olcf(
        client, enclave, token=s3m_tokens[enclave] or "no-token-public-endpoints-only"
    )

olcf = olcf_facilities[ACTIVE_ENCLAVE]
s3m_token = s3m_tokens[ACTIVE_ENCLAVE]
print(f"Using olcf-{ACTIVE_ENCLAVE}: {olcf.base_url}")

### 2c. Confirm the client uses API v2

`verify_api_version()` builds the URL amsc-client would call for the resource list, without sending it, and reads the API version from the server's public OpenAPI document. It stops with an error if the client would not call `/api/v2`.

In [ ]:
for facility in olcf_facilities.values():
    verify_api_version(facility)
    print()

### 2d. List resources

The cell also prints the requests the client actually sent to this enclave, as a second check that they went to `/api/v2`. The resource list comes from the discovery `register_olcf()` ran in Step 2b, so those are the requests shown.

In [ ]:
# List all available resources in the active enclave
resources = olcf.resources()

print(f"Available Resources ({len(resources)}):\n")
print(f"{'Name':12s}  {'Type':10s}  {'Status'}")
print(f"{'─'*12}  {'─'*10}  {'─'*12}")
for r in resources:
    rtype = getattr(r.resource_type, "value", r.resource_type)
    print(f"{r.name:12s}  {rtype:10s}  {r.status}")

print(f"\nRequests the client sent to olcf-{ACTIVE_ENCLAVE}:")
for request in recent_requests(10, base_url=olcf.base_url):
    print(f"  {request}")

### 2e. Choose a compute resource

Set `RESOURCE_NAME` to one of the names listed above (case-insensitive). It must belong to the active enclave:

- `open`: `"Defiant"`, `"Wombat"`, `"Quokka"`, `"Odo"`
- `moderate`: `"Frontier"`

Leave it as `None` to use the enclave default: Defiant for open, Frontier for moderate.

In [ ]:
# ── Compute resource ───────────────────────────────────────────────
RESOURCE_NAME = None        # e.g. "Odo"; None = enclave default
# ───────────────────────────────────────────────────────────────────

DEFAULT_RESOURCE = {"open": "Defiant", "moderate": "Frontier"}
name = RESOURCE_NAME or os.environ.get("OLCF_RESOURCE") or DEFAULT_RESOURCE[ACTIVE_ENCLAVE]
available = [r.name for r in resources]
if name.lower() not in (n.lower() for n in available):
    raise ValueError(f"{name!r} is not in the {ACTIVE_ENCLAVE} enclave. Choose one of: {available}")
resource = olcf.resource(name)

print(f"Resource: {resource.name}")
print(f"ID:       {resource.id}")
print(f"Type:     {getattr(resource.resource_type, 'value', resource.resource_type)}")
print(f"Status:   {resource.status}")

## Step 3: Check Your Identity (`whoami`)

`GET /api/v2/account/whoami` returns the identity behind an S3M token. It is the only Account endpoint S3M serves today. Use it to confirm a token works before submitting jobs.

- **200**: the token is valid. `username` is the account that jobs run as (your project's automation user, `prj###_auser`).
- **401**: the token is missing, expired, revoked, or was issued for the other enclave.

In [ ]:
import requests

for enclave, token in s3m_tokens.items():
    url = f"{ENCLAVES[enclave]}/api/v2/account/whoami"
    if not token:
        print(f"{enclave}: ⚪ no token configured — skipped {url}\n")
        continue
    try:
        identity = whoami(enclave, token)
    except requests.HTTPError as exc:
        print(f"{enclave}: ❌ token rejected (HTTP {exc.response.status_code}) by {url}\n")
        continue
    print(f"{enclave}: ✅ token valid ({url})")
    for key, value in identity.items():
        if not key.startswith("_"):
            print(f"    {key:12s} {value}")
    print()

## Step 4: Check for Active Incidents

Before submitting a job, it's good practice to check for any active outages or maintenance windows.

In [ ]:
incidents = olcf.incidents()
print(f"Total incidents: {len(incidents)}")

# Show the 5 most recent
if incidents:
    print(f"\nRecent incidents:")
    for inc in incidents[:5]:
        print( f' - {str(inc.last_modified):20} | {inc.name:25} -> {inc.resolution:20} ')

## Step 5: Configure and Submit a Job

Now we'll submit a simple "hello world" job to the resource chosen in Step 2e.

⚠️ Job submission requires a valid S3M token for your OLCF project, scoped to the active enclave. Check it in Step 3.

**Before running:** set the account, queue, and working directory for your project allocation, either as environment variables (`OLCF_ACCOUNT`, `OLCF_QUEUE`, `OLCF_WORK_DIR`) or by editing the cell below.

Queue options:
- Frontier: `batch` or `extended`
- Defiant: `batch-gpu`, `batch-cpu`, or `cron`
- Quokka: see the ACE Testbed documentation
- Wombat: `gh`, `Ampere-gpu`, `Ampere`, or `A64fx`
- Odo: `batch` or `cron`

In [ ]:
SUBMIT_JOB = False  # Set True only after reviewing all settings below.

# ── Job Configuration ──────────────────────────────────────────────
# Update these to match your OLCF project (or set the environment variables):

DEFAULT_QUEUE = {"Defiant": "batch-cpu", "Odo": "batch", "Frontier": "batch"}

OLCF_ACCOUNT = os.environ.get("OLCF_ACCOUNT", "<project-id>")             # e.g. abc123
OLCF_QUEUE   = os.environ.get("OLCF_QUEUE", DEFAULT_QUEUE.get(resource.name, "<queue>"))  # see list above
OUTPUT_DIR   = os.environ.get("OLCF_WORK_DIR", "<path/to/working-dir>")   # must exist and be writable

# NOTE: The IRI API submits jobs as the project automation user (prj###_auser).
# OUTPUT_DIR must be writable by that user, e.g.:
#   chmod g+rwx /lustre/polis/<project>/proj-shared/<working-dir>/

# ─────────────────────────────────────────────────────────────────

if SUBMIT_JOB and ("<" in OLCF_ACCOUNT or "<" in OUTPUT_DIR or "<" in OLCF_QUEUE):
    raise ValueError("Set OLCF_ACCOUNT, OLCF_QUEUE and OLCF_WORK_DIR (or edit this cell) before submitting.")
if SUBMIT_JOB and not s3m_token:
    raise ValueError(f"No S3M token for the {ACTIVE_ENCLAVE} enclave; see Step 2a.")

import datetime

def new_run_id() -> str:
    """Timestamp for a unique job name. amsc-client tracks jobs by name, so
    reusing one replaces the earlier job's entry (with a warning)."""
    return datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

print(f"Job names:   amsc-tutorial-<timestamp> (a new one for each submission)")
print(f"Enclave:     {ACTIVE_ENCLAVE}")
print(f"Resource:    {resource.name}")
print(f"Queue:       {OLCF_QUEUE}")
print(f"Account:     {OLCF_ACCOUNT}")
print(f"Output dir:  {OUTPUT_DIR}")

In [ ]:
if SUBMIT_JOB:
    # Submit a simple diagnostic job that runs for one minute.

    JOB_NODE_COUNT = 1
    JOB_RUNTIME_SECONDS = 60

    RUN_ID = new_run_id()
    JOB_NAME = f"amsc-tutorial-{RUN_ID}"   # unique per submission

    job_script = f"""
    set -e

    echo "Hello from AmSC Python Client!"
    echo "Run ID: {RUN_ID}"
    echo "Started: $(date -Is)"
    echo "Hostname: $(hostname)"
    echo "User: $(whoami 2>/dev/null || id -un)"
    echo "Working directory: $(pwd)"
    echo "Requested nodes: {JOB_NODE_COUNT}"
    echo

    echo "Scheduler environment:"
    for key in SLURM_JOB_ID SLURM_JOB_NAME SLURM_NNODES SLURM_JOB_NODELIST; do
        eval "value=\\${{$key:-}}"
        if [ -n "$value" ]; then
            echo "  $key=$value"
        fi
    done
    echo

    echo "Allocated nodes:"
    if [ -n "${{SLURM_JOB_NODELIST:-}}" ] && command -v scontrol >/dev/null 2>&1; then
        scontrol show hostnames "$SLURM_JOB_NODELIST"
    else
        hostname
    fi
    echo

    echo "Sleeping for {JOB_RUNTIME_SECONDS} seconds so the job remains visible in the queue."
    sleep {JOB_RUNTIME_SECONDS}
    echo "Finished: $(date -Is)"
    """.strip()

    job = resource.submit(
        executable="/bin/bash",
        arguments=["-lc", job_script],
        directory=OUTPUT_DIR,
        name=JOB_NAME,
        queue=OLCF_QUEUE,
        account=OLCF_ACCOUNT,
        duration=300,             # Wall time in seconds (5 minutes)
        nodes=JOB_NODE_COUNT,
        environment={
            "AMSC_RUN_ID": RUN_ID,
            "AMSC_SLEEP_SECONDS": str(JOB_RUNTIME_SECONDS),
        },
    )

    print(f"\n✅ Job submitted!")
    print(f"   Job name:  {JOB_NAME}")
    print(f"   Job ID:    {job.id}")
    print(f"   State:     {job.state}")
    print(f"   {job!r}")
else:
    print("SUBMIT_JOB=False: no scheduler job was submitted or monitored.")


## Step 6: Monitor the Job

The `Job` object is "self-aware" — it knows which facility and resource it belongs to,
and can refresh its own status from the API.

We can poll manually or use `job.wait()` to block until completion.

In [ ]:
if SUBMIT_JOB:
    # Manual polling — check status every 5 seconds

    POLL_TIMEOUT = 180  # seconds
    POLL_INTERVAL = 5   # seconds

    print(f"Polling job {job.id} (up to {POLL_TIMEOUT}s)...\n")
    start_time = time.time()

    while time.time() - start_time < POLL_TIMEOUT:
        elapsed = int(time.time() - start_time)
        try:
            current_status = job.status  # calls the API
            state = job.state
            print(f"  [{elapsed:3d}s] State: {state}")

            if job.is_terminal:
                exit_code = job.exit_code
                message = job.message
                print(f"\n✅ Job finished!")
                print(f"   Final state: {state}")
                if exit_code is not None:
                    print(f"   Exit code:   {exit_code}")
                if message:
                    print(f"   Message:     {message}")
                break
        except Exception as e:
            # Completed Slurm jobs disappear from the queue;
            # the API returns 400 "not found" in that case
            if "not found" in str(e).lower() or "400" in str(e):
                print(f"  [{elapsed:3d}s] Job no longer in scheduler queue (likely completed)")
                print(f"\n✅ Job completed (exited scheduler)")
                break
            else:
                print(f"  [{elapsed:3d}s] Status check error: {e}")

        time.sleep(POLL_INTERVAL)
    else:
        print(f"\n⏰ Timed out after {POLL_TIMEOUT}s (job may still be running)")
        print(f"   Last known state: {job.state}")
else:
    print("SUBMIT_JOB=False: no scheduler job was submitted or monitored.")


## Alternative: Using `job.wait()`

Instead of manual polling, you can use the built-in `wait()` method
which blocks until the job reaches a terminal state:

In [ ]:
if SUBMIT_JOB:
    JOB_NODE_COUNT = 1
    JOB_RUNTIME_SECONDS = 60

    RUN_ID = new_run_id()
    JOB_NAME_WAIT = f"amsc-tutorial-wait-{RUN_ID}"   # unique per submission

    job_script = f"""
    set -e

    echo "Hello from AmSC Python Client!"
    echo "Run ID: {RUN_ID}"
    echo "Started: $(date -Is)"
    echo "Hostname: $(hostname)"
    echo "User: $(whoami 2>/dev/null || id -un)"
    echo "Working directory: $(pwd)"
    echo "Requested nodes: {JOB_NODE_COUNT}"
    echo

    echo "Scheduler environment:"
    for key in SLURM_JOB_ID SLURM_JOB_NAME SLURM_NNODES SLURM_JOB_NODELIST; do
        eval "value=\\${{$key:-}}"
        if [ -n "$value" ]; then
            echo "  $key=$value"
        fi
    done
    echo

    echo "Allocated nodes:"
    if [ -n "${{SLURM_JOB_NODELIST:-}}" ] && command -v scontrol >/dev/null 2>&1; then
        scontrol show hostnames "$SLURM_JOB_NODELIST"
    else
        hostname
    fi
    echo

    echo "Sleeping for {JOB_RUNTIME_SECONDS} seconds so the job remains visible in the queue."
    sleep {JOB_RUNTIME_SECONDS}
    echo "Finished: $(date -Is)"
    """.strip()

    job2 = resource.submit(
        executable="/bin/bash",
        arguments=["-lc", job_script],
        directory=OUTPUT_DIR,
        name=JOB_NAME_WAIT,
        queue=OLCF_QUEUE,
        account=OLCF_ACCOUNT,
        duration=300,             # Wall time in seconds (5 minutes)
        nodes=JOB_NODE_COUNT,
        environment={
            "AMSC_RUN_ID": RUN_ID,
            "AMSC_SLEEP_SECONDS": str(JOB_RUNTIME_SECONDS),
        },
    )

    print(f"\n✅ Job submitted!")
    print(f"   Job name:  {JOB_NAME_WAIT}")
    print(f"   Job ID:    {job2.id}")
    print(f"   State:     {job2.state}")
    print(f"   {job2!r}")

    try:
        job2.wait(timeout=180, poll_interval=5)
        print(f"✅ Job completed: state={job2.state}, exit_code={job2.exit_code}")
    except TimeoutError:
        print(f"⏳ Job still running after timeout (state: {job2.state})")
else:
    print("SUBMIT_JOB=False: no scheduler job was submitted or monitored.")


## Step 7: APIs Not Yet Available in S3M

The ALCF and NERSC tutorials read job output with the **Filesystem API**. S3M does not serve it yet, so check job output on the OLCF filesystem directly.

| IRI API group | S3M status |
|---------------|------------|
| Facility, Status, Compute | ✅ Available (v2) |
| Account | Only `GET /api/v2/account/whoami` |
| Filesystem | Not yet available |
| Task | Not yet available |
| Storage | Not yet available |

## Summary

This tutorial showed how to:

| Step | API Call | Auth Required? |
|------|---------|----------------|
| Load a token | `resolve_token("open", "env" / "file" / "prompt")` | — |
| Connect to OLCF | `register_olcf(client, "open")` | No |
| Confirm API v2 | `verify_api_version(olcf)` | No |
| Check identity | `whoami("open", token)` → `GET /api/v2/account/whoami` | Yes (S3M Token) |
| List resources | `olcf.resources()` | No |
| Get resource details | `olcf.resource("Defiant")` | No |
| Check incidents | `olcf.incidents()` | No |
| Submit a job | `resource.submit(...)` | Yes (S3M Token) |
| Check job status | `job.status` | Yes (S3M Token) |
| Wait for completion | `job.wait(timeout=180)` | Yes (S3M Token) |

**Key concepts:**
- **Enclave-scoped access**: open and moderate are separate S3M deployments, each with its own token
- **IRI v2**: `olcf_s3m.register_olcf()` routes amsc-client's facility calls to S3M's v2 API
- **Resource-scoped compute**: Jobs are submitted via the resource object (`resource.submit(...)`) rather than a separate compute client
- **Self-aware jobs**: The `Job` object tracks its own facility and resource, and can refresh status, wait, or cancel itself
- **Jobs run as the project automation user (prj###_auser)**: make sure the working directory has the correct permissions

## Troubleshooting

### 401 Unauthorized

- Run the `whoami` check in Step 3. A rejected token is expired, revoked, or was issued for the other enclave.
- Make sure the token is scoped to the project in `OLCF_ACCOUNT` and allows compute access.

### Job submission errors

- **Wrong queue**: queue names differ per resource (see Step 5).
- **Permission denied in the working directory**: grant the project automation user access (`chmod g+rwx <dir>`).

OLCF is actively improving IRI/S3M error tracking and messaging. Please contact OLCF Help (help@olcf.ornl.gov) to provide feedback and request assistance.